# 🛒 Superstore — Pipeline Data Engineering
**Stack :** Python · pandas · SQLAlchemy · PostgreSQL

> Ce notebook lit le fichier CSV, prépare les données, les découpe en tables normalisées, puis les insère dans PostgreSQL.

---
## Cellule 1 — Importation des bibliothèques

In [65]:
import pandas as pd
from sqlalchemy import create_engine

---
## Cellule 2 — Chargement du dataset

In [66]:
df = pd.read_csv("a.csv", encoding="latin1")

df.head()

,row_id,order_id,order_date,ship_date,ship_mode,customer_id,customer_name,segment,country,city,...,sales,order_year,order_month,ship_year,ship_month,order_quarter,cost,profit,profit_ratio,div
0,1,CA-2017-152156,2017-11-08,2017-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,261.9600,2017,11,2017,11,4,157.1760,104.7840,40.0,3
1,2,CA-2017-152156,2017-11-08,2017-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,731.9400,2017,11,2017,11,4,439.1640,292.7760,40.0,3
2,3,CA-2017-138688,2017-06-12,2017-06-16,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,14.6200,2017,6,2017,6,2,8.7720,5.8480,40.0,4
3,4,US-2016-108966,2016-10-11,2016-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,957.5775,2016,10,2016,10,4,574.5465,383.0310,40.0,7
4,5,US-2016-108966,2016-10-11,2016-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,22.3680,2016,10,2016,10,4,13.4208,8.9472,40.0,7


---
## Cellule 3 — Nettoyage de base

In [67]:
df.columns = df.columns.str.strip().str.replace("-", "_", regex=False)

df["order_date"] = pd.to_datetime(df["order_date"])
df["ship_date"]  = pd.to_datetime(df["ship_date"])

print("✅ Nettoyage terminé")
print("Dimensions du dataset :", df.shape)
df.columns

✅ Nettoyage terminé
Dimensions du dataset : (9776, 27)


Index(['row_id', 'order_id', 'order_date', 'ship_date', 'ship_mode',
       'customer_id', 'customer_name', 'segment', 'country', 'city', 'state',
       'postal_code', 'region', 'product_id', 'category', 'sub_category',
       'product_name', 'sales', 'order_year', 'order_month', 'ship_year',
       'ship_month', 'order_quarter', 'cost', 'profit', 'profit_ratio', 'div'],
      dtype='object')

---
## Cellule 4 — Table `regions`

In [68]:
regions = df[["postal_code", "city", "state", "region", "country"]].copy()

regions = regions.drop_duplicates(subset=["postal_code"])
regions = regions.reset_index(drop=True)

print("Nombre de régions :", len(regions))
regions.head()

Nombre de régions : 626


,postal_code,city,state,region,country
0,42420,Henderson,Kentucky,South,United States
1,90036,Los Angeles,California,West,United States
2,33311,Fort Lauderdale,Florida,South,United States
3,90032,Los Angeles,California,West,United States
4,28027,Concord,North Carolina,South,United States


---
## Cellule 5 — Table `customers`

In [69]:
customers = df[["customer_id", "customer_name", "segment", "postal_code"]].copy()

customers = customers.drop_duplicates(subset=["customer_id"])
customers = customers.reset_index(drop=True)

print("Nombre de clients :", len(customers))
customers.head()

Nombre de clients : 793


,customer_id,customer_name,segment,postal_code
0,CG-12520,Claire Gute,Consumer,42420
1,DV-13045,Darrin Van Huff,Corporate,90036
2,SO-20335,Sean O'Donnell,Consumer,33311
3,BH-11710,Brosina Hoffman,Consumer,90032
4,AA-10480,Andrew Allen,Consumer,28027


---
## Cellule 6 — Table `categories`

In [70]:
categories = df[["category"]].drop_duplicates().copy()
categories = categories.reset_index(drop=True)

categories.index = categories.index + 1
categories = categories.reset_index()
categories.columns = ["category_id", "category_name"]

print("Catégories :")
categories

Catégories :


,category_id,category_name
0,1,Furniture
1,2,Office Supplies
2,3,Technology


---
## Cellule 7 — Table `products`

In [71]:
products = df[["product_id", "product_name", "sub_category", "category"]].copy()
products = products.drop_duplicates(subset=["product_id"])

products = products.merge(
    categories.rename(columns={"category_name": "category"}),
    on="category",
    how="left"
)
products = products.drop(columns=["category"])
products = products.reset_index(drop=True)

print("Nombre de produits :", len(products))
products.head()

Nombre de produits : 1859


,product_id,product_name,sub_category,category_id
0,FUR-BO-10001798,Bush Somerset Collection Bookcase,Bookcases,1
1,FUR-CH-10000454,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",Chairs,1
2,OFF-LA-10000240,Self-Adhesive Address Labels for Typewriters b...,Labels,2
3,FUR-TA-10000577,Bretford CR4500 Series Slim Rectangular Table,Tables,1
4,OFF-ST-10000760,Eldon Fold 'N Roll Cart System,Storage,2


---
## Cellule 8 — Table `orders`

In [72]:
orders = df[["order_id", "order_date", "ship_date", "ship_mode", "customer_id"]].copy()

orders = orders.drop_duplicates(subset=["order_id"])
orders = orders.reset_index(drop=True)

print("Nombre de commandes :", len(orders))
orders.head()

Nombre de commandes : 4915


,order_id,order_date,ship_date,ship_mode,customer_id
0,CA-2017-152156,2017-11-08,2017-11-11,Second Class,CG-12520
1,CA-2017-138688,2017-06-12,2017-06-16,Second Class,DV-13045
2,US-2016-108966,2016-10-11,2016-10-18,Standard Class,SO-20335
3,CA-2015-115812,2015-06-09,2015-06-14,Standard Class,BH-11710
4,CA-2018-114412,2018-04-15,2018-04-20,Standard Class,AA-10480


---
## Cellule 9 — Table `order_details`

In [73]:
order_details = df[["order_id", "product_id", "sales", "cost", "profit"]].copy()

order_details = order_details.drop_duplicates(subset=["order_id", "product_id"])
order_details = order_details.reset_index(drop=True)

print("Nombre de lignes de détail :", len(order_details))
order_details.head()

Nombre de lignes de détail : 9769


,order_id,product_id,sales,cost,profit
0,CA-2017-152156,FUR-BO-10001798,261.9600,157.1760,104.7840
1,CA-2017-152156,FUR-CH-10000454,731.9400,439.1640,292.7760
2,CA-2017-138688,OFF-LA-10000240,14.6200,8.7720,5.8480
3,US-2016-108966,FUR-TA-10000577,957.5775,574.5465,383.0310
4,US-2016-108966,OFF-ST-10000760,22.3680,13.4208,8.9472


---
## Cellule 10 — Connexion à PostgreSQL

In [74]:
password = "1234"
host     = "localhost"
port     = "5432"
database = "superstore_db_1"

engine = create_engine(
    f"postgresql+psycopg2://postgres:{password}@{host}:{port}/{database}",
    connect_args={"client_encoding": "utf8"}
)

print("Connexion à PostgreSQL réussie !")

Connexion à PostgreSQL réussie !


---
## Cellule 11 — Insertion des données dans PostgreSQL

In [75]:
with engine.begin() as conn:

    regions.to_sql("regions", conn, if_exists="append", index=False)

    customers.to_sql("customers", conn, if_exists="append", index=False)

    categories.to_sql("categories", conn, if_exists="append", index=False)

    products.to_sql("products", conn, if_exists="append", index=False)

    orders.to_sql("orders", conn, if_exists="append", index=False)

    order_details.to_sql("order_details", conn, if_exists="append", index=False)

print("\n Pipeline terminé — toutes les tables sont chargées dans PostgreSQL !")



 Pipeline terminé — toutes les tables sont chargées dans PostgreSQL !
